# Parameter tuning for deterministic optimization.

This notebook loads the results of different deterinistic optimization algorithms applied to various scheduling problems,
performs summary statistics, and conducts significance testing on the results. It generates a LaTeX table summarizing the findings to be reported in the article.

Instructions:

- Run cells below with b_with_tardiness set to True or False to compare results with and without tardiness in the objective function.
- It will save .tex file with the results table in the `latex` directory.
- Copy the generated .tex files to Overleaf results folder.

In [6]:
from pathlib import Path

import pandas as pd

# Data loading

In [7]:
b_with_tardiness = True

DATA_DIR = Path("../results")
all_csv_files = """FlexibleJobShop_2025-06-14_18-03-57.csv
FlexibleJobShopFullStoch_2025-06-14_18-04-15.csv
HybridFlowShop_2025-06-14_18-04-25.csv
HybridFlowShopFullStoch_2025-06-14_18-04-36.csv
JobShop_2025-06-14_18-05-00.csv
JobShopFullStoch_2025-06-14_18-04-53.csv
OpenShop_2025-06-14_18-03-30.csv
OpenShopFullStoch_2025-06-14_18-03-54.csv
ParallelMachines_2025-06-14_18-05-20.csv
ParallelMachinesFullStoch_2025-06-14_18-05-12.csv
HybridFlowShopFullStochNoTard_2025-06-15_08-47-09.csv
HybridFlowShopNoTard_2025-06-15_08-46-51.csv
ParallelMachinesFullStochNoTard_2025-06-15_08-47-24.csv
ParallelMachinesNoTard_2025-06-15_08-47-22.csv
OpenShopFullStochNoTard_2025-06-15_08-46-47.csv
OpenShopNoTard_2025-06-15_08-46-47.csv
JobShopFullStochNoTard_2025-06-15_08-47-09.csv
FlexibleJobShopFullStochNoTard_2025-06-15_08-46-47.csv
JobShopNoTard_2025-06-15_08-47-09.csv
FlexibleJobShopNoTard_2025-06-15_08-46-47.csv""".split('\n')
# No tardiness had no +1 in all means, which made comparison hard
# all_csv_files = """FlexibleJobShop_2025-06-14_18-03-57.csv
# FlexibleJobShopFullStoch_2025-06-14_18-04-15.csv
# FlexibleJobShopFullStochNoTard_2025-06-14_18-04-16.csv
# FlexibleJobShopNoTard_2025-06-14_18-04-10.csv
# HybridFlowShop_2025-06-14_18-04-25.csv
# HybridFlowShopFullStoch_2025-06-14_18-04-36.csv
# HybridFlowShopFullStochNoTard_2025-06-14_18-04-34.csv
# HybridFlowShopNoTard_2025-06-14_18-04-37.csv
# JobShop_2025-06-14_18-05-00.csv
# JobShopFullStoch_2025-06-14_18-04-53.csv
# JobShopFullStochNoTard_2025-06-14_18-04-52.csv
# JobShopNoTard_2025-06-14_18-04-53.csv
# OpenShop_2025-06-14_18-03-30.csv
# OpenShopFullStoch_2025-06-14_18-03-54.csv
# OpenShopFullStochNoTard_2025-06-14_18-03-54.csv
# OpenShopNoTard_2025-06-14_18-03-51.csv
# ParallelMachines_2025-06-14_18-05-20.csv
# ParallelMachinesFullStoch_2025-06-14_18-05-12.csv
# ParallelMachinesFullStochNoTard_2025-06-14_18-05-21.csv
# ParallelMachinesNoTard_2025-06-14_18-05-22.csv""".split('\n')
problem_order = ['OpenShop', 'ParallelMachines', 'HybridFlowShop', 'JobShop', 'FlexibleJobShop']
def find_with_part(file_names, part):
    """Find all file names that contain a specific part."""
    return [name for name in file_names if part == name.split('_')[0]][0]

if b_with_tardiness:
    file_names_per_exp_group = {
        "p = 0.3": [
            find_with_part(all_csv_files, problem_name)                
            for problem_name in problem_order
            ],
        "p = 1.0": [
            find_with_part(all_csv_files, problem_name + "FullStoch")
            for problem_name in problem_order
            ],
        }  # the key denotes a group of experiments
else:
    file_names_per_exp_group = {
        "p = 0.3":[
            find_with_part(all_csv_files, problem_name + "NoTard")
            for problem_name in problem_order
            ],
        "p = 1.0": [
            find_with_part(all_csv_files, problem_name + "FullStochNoTard")
            for problem_name in problem_order
            ],
        }  # without tardiness
print(file_names_per_exp_group)
dfs = {}
for exp_group, file_names in file_names_per_exp_group.items():
    dfs[exp_group] = {}
    for file_name in file_names:
        dfs[exp_group][file_name] = pd.read_csv(DATA_DIR / file_name)
        # display(dfs[exp_group][file_name].head(4))


# Summary statistics

In [8]:
numeric_cols = [
    "mean",
    "DCOP_objective",
]
custom_simh_order = ["det_opt_1", "det_opt_2", "det_opt_3", "det_opt_4"]

def get_problem_name_from_file_name(file_name):
    """Extract the problem name from the file name."""
    first_part = file_name.split("_")[0]
    if "FullStochNoTard" in first_part:
        return first_part.replace("FullStochNoTard", "")
    elif "FullStoch" in first_part:
        return first_part.replace("FullStoch", "")
    elif "NoTard" in first_part:
        return first_part.replace("NoTard", "")
    else:
        return first_part

# Create a dictionary to hold the summaries for each problem
summaries_dfs = {}
for exp_group, dfs_exp_group in dfs.items():
    for file_name, df in dfs_exp_group.items():

        # Convert numeric columns to proper types (in case they were strings)
        df[numeric_cols] = df[numeric_cols].apply(
            pd.to_numeric,
            errors="coerce"
        )

        # Group by simheuristic and take the mean of selected columns
        summary = (
            df.groupby("simheuristic")[numeric_cols]
            .mean()
            .round(2)
            .reindex(custom_simh_order)
        )

        # Store the summary in a dictionary
        problem_name = get_problem_name_from_file_name(file_name)
        if problem_name not in summaries_dfs:
            summaries_dfs[problem_name] = {}
        summaries_dfs[problem_name][exp_group] = summary

# Create a dictionary to hold concatenated summaries with MultiIndex columns
combined_summaries = {}

for problem_name, exp_group_summaries in summaries_dfs.items():
    dfs_to_concat = []

    for exp_group, df_summary in exp_group_summaries.items():
        # Create a MultiIndex for the columns: (exp_group, column_name)
        df_summary.columns = pd.MultiIndex.from_product(
            [[exp_group], df_summary.columns]
        )
        dfs_to_concat.append(df_summary)

    # Concatenate along columns (axis=1)
    combined_summary = pd.concat(dfs_to_concat, axis=1)

    combined_summaries[problem_name] = combined_summary
    # display(combined_summary)


# Significance testing

In [9]:
from scipy.stats import ttest_rel

# Your significance level
alpha = 0.05

# Find Sidak corrected (cor) significance level
n_tests = 3
alpha_cor = 1 - (1 - alpha) ** (1 / n_tests)

# Find dict tha for each methods gives the signific. outperformed methods
method_significant_better_than = {}
best_method_percentage = {}
for exp_group, dfs_exp_group in dfs.items():
    method_significant_better_than[exp_group] = {}
    best_method_percentage[exp_group] = {}
    for file_name, df in dfs_exp_group.items():
        problem_name = get_problem_name_from_file_name(file_name)
        method_significant_better_than[exp_group][problem_name] = {}
        best_method_percentage[exp_group][problem_name] = {}
        for metric in ["mean", "DCOP_objective"]:
            # Prepare data
            display(df)
            pivot = df.pivot(index="seed", columns="simheuristic", values=metric)
            methods = pivot.columns
            results = pd.DataFrame(index=methods, columns=methods)
        
            # Pairwise t-tests
            df_means = summaries_dfs[problem_name][exp_group][(exp_group, metric)]
            method_significant_better_than[exp_group][problem_name][metric] = {m: [] for m in methods}
            for idx, m1 in enumerate(custom_simh_order):
                for m2 in custom_simh_order[idx + 1:]:
                    stat, p = ttest_rel(pivot[m1], pivot[m2])
                    if p < alpha_cor:
                        # m1 and m2 results are significantly different
                        if df_means.loc[m1] < df_means.loc[m2]:
                            method_significant_better_than[exp_group][problem_name][metric][m1].append(m2)    
                        else:
                            method_significant_better_than[exp_group][problem_name][metric][m2].append(m1)
                            
            # Check how often each method is best over all seeds
            best_method_percentage[exp_group][problem_name][metric] = {m: 0 for m in methods}
            num_exps = len(pivot.index)
            for seed in pivot.index:
                min_value = pivot.loc[seed].min()  # Get the minimum value for this seed
                best_methods = pivot.loc[seed][pivot.loc[seed] == min_value].index  # Get all methods with the minimum value
                for best_method in best_methods:
                    best_method_percentage[exp_group][problem_name][metric][best_method] += 100 * 1 / num_exps
# print(exp_group)
# print(method_significant_better_than[exp_group])
# print(pivot)
# print(best_method_counter[exp_group][problem_name][metric])

# Latex table

In [10]:
# Mapping for better LaTeX names
label_opt_approach = "\\textbf{Optimization approach}"
label_mean_objective = "\\textbf{Mean objective}"
label_objective = "\\textbf{Objective}"
label_problem = "\\textbf{Problem}"
latex_name_mapping = {
    "mean": label_mean_objective,
    "DCOP_objective": label_objective,
    "det_opt_1": "Deterministic opt. 1",
    "det_opt_2": "Deterministic opt. 2",
    "det_opt_3": "Deterministic opt. 3",
    "det_opt_4": "Deterministic opt. 4",
    "dyn_simh": "Dynamic simheuristic",
    "sim_last": "Simulate-last heuristic",
    "std_simh": "Standard simheuristic",
    "simheuristic": label_opt_approach,
    "FlexibleJobShop": "FJS",
    "HybridFlowShop": "HFS",
    "JobShop": "JS",
    "ParallelMachines": "PM",
    "OpenShop": "OS",
    "p = 0.3": "\\underline{\\textbf{30\\% of durations random}}",
    "p = 1.0": "\\underline{\\textbf{All durations random}}",
    # Add more mappings as needed
    }

# Merge all problem dataframes and add a 'Problem' column
merged_df_list = []
for problem_name, combined_df in combined_summaries.items():
    # Add a column for the problem name
    combined_df[label_problem] = problem_name
    merged_df_list.append(combined_df)

# Concatenate all dataframes into a single one
df_merged = pd.concat(merged_df_list)
df_merged.reset_index(inplace=True)

# Now apply transformations on the merged dataframe
df_latex = df_merged.copy()

# Rename columns (first and second level)
df_latex.columns = pd.MultiIndex.from_tuples([
    (
        latex_name_mapping.get(exp_group, exp_group),   # First level
        latex_name_mapping.get(metric, metric)          # Second level
        )
    for exp_group, metric in df_latex.columns
    ])

# Create MultiIndex
df_latex.set_index([label_problem, label_opt_approach], inplace=True)

# Rename rows (index) using the latex_name_mapping dictionary
df_latex.index = df_latex.index.set_levels([
    df_latex.index.levels[0].map(lambda x: latex_name_mapping.get(x, x)),  # Rename 'Problem' index
    df_latex.index.levels[1].map(lambda x: latex_name_mapping.get(x, x))   # Rename 'Optimization approach' index
    ])

# Rename index names as well
df_latex.index.names = [latex_name_mapping.get(name, name) for name in df_latex.index.names]

# Bold best (lowest) values in each column within each problem
for metric in df_latex.columns:
    df_latex[metric] = df_latex.groupby(level=0)[[metric]].transform(
        lambda group: [
            f"\\textbf{{{val:.2f}}}" if val == group.min() else f"{val:.2f}"
            for val in group
            ]
        )

# Add superscripts for methods that are significantly better
roman_numerals = ["i", "ii", "iii", "iv", "v", "vi", "vii", "viii", "ix", "x"]
numerals = ['1', '2', '3', '4', '5', '6', '7', '8', '9', '10']
assert len(roman_numerals) >= len(custom_simh_order)
method_symbols = {
    simh_name: numerals[idx]
    for idx, simh_name in enumerate(custom_simh_order)
    }
inverse_latex_mapping = {v: k for k, v in latex_name_mapping.items()}
for (exp_group, metric) in df_latex.columns:
    for (problem, method) in df_latex.index:
        # Get LaTeX-formatted value (already may contain \textbf{})
        value = df_latex.loc[(problem, method), (exp_group, metric)]

        # Reverse map to original keys
        raw_problem = inverse_latex_mapping[problem]
        raw_method = inverse_latex_mapping[method]
        raw_exp_group = inverse_latex_mapping[exp_group]
        raw_metric = inverse_latex_mapping[metric]

        # Get list of worse methods
        worse_methods = method_significant_better_than[raw_exp_group][raw_problem][raw_metric][raw_method]
        if len(worse_methods) > 0:
            # Add superscript
            symbols = ",".join(method_symbols[m] for m in worse_methods)
            value = value + "$^{(" + symbols + ")}$"
            df_latex.loc[(problem, method), (exp_group, metric)] = value
                
        # Add percentage how often the best
        perc = best_method_percentage[raw_exp_group][raw_problem][raw_metric][raw_method]
        if value.endswith("}$"):
            value = value[:-1]
            value = value + "_{" + f"({perc:.0f}\\%)" + "}$"
        else:
            value = value + "$_{" + f"({perc:.0f}\\%)" + "}$"
        df_latex.loc[(problem, method), (exp_group, metric)] = value
        
# Export to LaTeX for the merged table
if b_with_tardiness:
    caption = "Summary results parameter tuning deterministic optimization for all problems with tardiness objective."
    label = "tab:param_tuning_det_opt_all_problems_with_tardiness"
else:
    caption = "Summary results parameter tuning deterministic optimization for all problems without tardiness objective."
    label = "tab:param_tuning_det_opt_all_problems_without_tardiness"
caption += " Best values marked in \\textbf{bold}. If a method is significantly better than others, it is marked with superscript in parentheses: "
# markings = ', '.join('$' + method_symbols[simh_name] + '$' + " - " + latex_name_mapping[simh_name].lower() for simh_name in custom_simh_order) 
markings = '$i$ stands for deterministic optimization $i$'
caption += markings + '.'
caption += ' The percentage in parentheses indicates how often the method was the best across all seeds.'

# Export the LaTeX table
latex_table = df_latex.to_latex(
    index=True,
    multirow=True,
    multicolumn=True,
    multicolumn_format='c',
    escape=False,
    caption=caption,
    column_format='ll|cc|cc',
    label=label
    )

# Correctly center the table
latex_table = latex_table.replace("\\begin{table}", "\\begin{table}[ht!]\n\\centering")
latex_table = latex_table.replace("\multicolumn{2}{c}", "\multicolumn{2}{c|}", 1)

# Manual string replacement for multirow and optimization approaches
old_part = f" &  & {label_mean_objective}"
new_part = f"{label_problem} & {label_opt_approach} & {label_mean_objective}"
latex_table = latex_table.replace(old_part, new_part, 1)
remove_part = f"{label_problem} & {label_opt_approach} &  &  &  &  \\\\\n"
latex_table = latex_table.replace(remove_part, "", 1)

# Remove the extra line at the end of the table
latex_table = latex_table.replace("\\cline{1-6}\n\\bottomrule", "\\bottomrule")

# # Print final LaTeX code
# print(f"% === LaTeX table for all problems ===")
# print(latex_table)
print(df_latex)
print(caption) 

# Save to .tex file
if b_with_tardiness:
    tex_name = "param_tuning_det_opt_with_tardiness"
else:
    tex_name = "param_tuning_det_opt_without_tardiness"
with open(f"latex/{tex_name}.tex", "w") as f:
    f.write(latex_table)
    print(f"Latex table saved to latex/{tex_name}.tex")
